# One Look · 06 · Piste VLM, affinage LoRA (Kaggle, GPU)

Même expérience que `05_colab_vlm.ipynb`, sur Kaggle : Colab gratuit a coupé l'accès au GPU (quota d'utilisation) avant l'affinage, le 13 nov. Kaggle donne 30 h de GPU par semaine, avec un compteur visible.

Le **zéro-shot n'est pas refait** : il a tourné sur Colab le 12 nov (256M et 500M répondent « incendie » à presque tout, voir `docs/results.md`). Ce notebook lance seulement :

| Run | Modèle | Ce qu'on mesure |
|---|---|---|
| `smolvlm256_lora` | SmolVLM 256M affiné par LoRA | ce qu'il apprend des 1 568 tuiles (lancé en premier) |
| `smolvlm500_lora` | SmolVLM 500M affiné par LoRA | l'effet de la taille |
| `smolvlm500_lora_4bits` | même adaptateur, poids en 4 bits | la version « compressée » |

Chaque run est envoyé sur Hugging Face dès qu'il est fini : une coupure ne perd que l'étape en cours, et une relance saute les runs déjà faits.

**Avant de lancer** (panneau de droite, *Settings*) :
1. *Accelerator* : **GPU T4 x2** (pas P100 : trop ancien pour le PyTorch du projet) ;
2. *Internet* : **on** (Kaggle demande une fois de vérifier le numéro de téléphone) ;
3. *Add-ons › Secrets* : `HF_TOKEN` (jeton Hugging Face en écriture), case cochée pour ce notebook.

Puis *Save Version › Save & Run All (Commit)* : le calcul continue même navigateur fermé, la sortie reste dans l'onglet *Versions*.

In [ ]:
HF_DATA = "Elfried9684/onelook-alert-data"
HF_RUNS = "Elfried9684/onelook-alert-runs"
REPO_URL = "https://github.com/elfried96/onelook-alert_Elf.git"
WORKDIR = "/tmp/onelook-alert_Elf"  # hors de /kaggle/working (20 Go, sauvegardé) : le paquet fait 7 Go
MODELS = {  # nom court -> modèle Hugging Face (licence Apache 2.0)
    "smolvlm256": "HuggingFaceTB/SmolVLM-256M-Instruct",
    "smolvlm500": "HuggingFaceTB/SmolVLM-500M-Instruct",
}
TRAIN = "--epochs 2 --batch 2 --accum 4"  # 2 tuiles par passe, gradients cumulés sur 4 : lot effectif de 8 (mémoire du T4)

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


def done(name):
    """Run déjà évalué sur le test (relance du notebook) : on ne le refait pas."""
    return os.path.exists(f"runs/{name}/eval_test_model.json")


def check(step):
    """Arrête le notebook à la première commande en échec, au lieu d'enchaîner des erreurs en cascade."""
    if _exit_code != 0:
        raise RuntimeError(f"Échec : {step}. L'erreur complète est juste au-dessus.")


def save():
    """Envoie les runs VLM déjà faits sur le dépôt privé : rien n'est perdu si une étape suivante échoue."""
    !uv run hf repos create {HF_RUNS} --type model --private --exist-ok --quiet
    !uv run hf upload {HF_RUNS} runs runs --include "smolvlm*/**" --quiet

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : Settings › Accelerator › GPU T4 x2"
!df -h /tmp /kaggle/working | tail -n 2

## 1. Code, environnement et données

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # un seul des deux T4 : le code tourne sur un GPU
if not os.path.exists(WORKDIR):
    !git clone -q {REPO_URL} {WORKDIR}
else:
    !git -C {WORKDIR} pull -q
%cd {WORKDIR}
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:/usr/local/bin:" + os.environ["PATH"]
!uv sync --frozen --group vlm
check("installation")
!git log --oneline -1
!uv run hf download {HF_DATA} --repo-type dataset --local-dir data/pack --exclude "pack12/*" --quiet
!uv run python scripts/check_pack.py

Essai éclair (deux pas d'affinage, 16 tuiles de validation) : vérifie la chaîne LoRA sur GPU avant les vrais entraînements. S'il échoue, copier l'erreur complète.

In [ ]:
!uv run --group vlm python scripts/vlm.py train --name essai_vlm --smoke {TRAIN}
check("essai éclair")
!rm -rf runs/essai_vlm

## 2. Affinage LoRA sur l'entraînement

256M d'abord, puis 500M. Compter de 20 à 40 minutes par modèle, plus l'évaluation (val puis test, avec la phrase rédigée).

In [ ]:
for short, model in MODELS.items():
    if done(f"{short}_lora"):
        continue
    !uv run --group vlm python scripts/vlm.py train --name {short}_lora --model {model} {TRAIN}
    check(f"affinage {short}")
    for split in ("val", "test"):
        !uv run --group vlm python scripts/vlm.py evaluate --name {short}_lora --model {model} --adapter runs/{short}_lora/adapter --split {split} --batch 16
        check(f"{short}_lora {split}")
    save()  # envoyé tout de suite : rien de perdu si la suite est coupée

## 4. Version compressée : poids en 4 bits

In [ ]:
model = MODELS["smolvlm500"]
if os.path.exists("runs/smolvlm500_lora/adapter") and not done("smolvlm500_lora_4bits"):
    for split in ("val", "test"):
        !uv run --group vlm python scripts/vlm.py evaluate --name smolvlm500_lora_4bits --model {model} --adapter runs/smolvlm500_lora/adapter --bits 4 --split {split} --batch 16
        check(f"4 bits {split}")
save()

## 5. Bilan sur le test

Zéro-shot (Colab, 12 nov, test) pour comparer : 256M 0 % · 98 % de rappel, 92 % de fausses alertes ; 500M 0 % · 100 %, 99 %.

In [ ]:
import json, glob, pandas as pd
rows = []
for path in sorted(glob.glob("runs/smolvlm*/eval_test_model.json")):
    r = json.load(open(path))
    rows.append({"run": path.split("/")[1], "rappel inond.": r["rappel_inondation"], "rappel incendie": r["rappel_incendie"],
                 "fausses alertes": r["fausses_alertes"], "fausses alertes eau": r["fausses_alertes_eau"],
                 "précision inond.": r["precision_inondation"], "précision incendie": r["precision_incendie"],
                 "taille Mo": r["taille_mo"]})
pd.DataFrame(rows).set_index("run").round(3)

In [ ]:
# Ce que le VLM affiné écrit : une réponse par décision, et les pièges d'eau
run = next(r for r in ("smolvlm500_lora", "smolvlm256_lora") if os.path.exists(f"runs/{r}/decisions_test_model.csv"))
print(run)
d = pd.read_csv(f"runs/{run}/decisions_test_model.csv")
index = pd.read_csv("data/pack/index.csv").set_index("id")
d["categorie"] = d["id"].map(index["category"])
pd.set_option("display.max_colwidth", 120)
display(d.groupby("decision").head(3)[["label", "decision", "categorie", "reponse"]])
display(d[d["categorie"].isin(["ocean", "riviere", "lac", "reservoir", "rizieres", "estuaire", "littoral"])][["label", "decision", "categorie", "reponse"]].head(12))

## 6. Ranger les résultats

Dépôt **privé** Hugging Face, à côté des runs du réseau (adaptateurs LoRA de quelques Mo, décisions, mesures). Une copie reste aussi dans la sortie de la version Kaggle.

In [ ]:
save()
print(f"https://huggingface.co/{HF_RUNS}/tree/main/runs")
!mkdir -p /kaggle/working/runs && cp -r runs/smolvlm* /kaggle/working/runs/

Ensuite, en local : `uv run hf download Elfried9684/onelook-alert-runs --include "runs/smolvlm*" --local-dir .`, puis le temps sur un cœur de processeur : `scripts/vlm.py evaluate ... --split test --limit 5 --timing 10` dans un dossier d'essai.